# Build the tillage montages (Colab)

**Self-contained**: the pipeline lives in one cell, so the smoke test and the full run
use the same code. (The previous version imported a module and then shelled out to a
script — the `MODE` switch silently did nothing and the full run used different settings.)

This is **not** the Earth Engine Code Editor (that's JavaScript and has no filesystem —
it cannot montage, overlay, or write CSVs). Run top to bottom.


## 1. Install


In [ ]:
!pip install -q earthengine-api pyproj pillow requests
print('deps ready')


## 2. Authenticate


In [ ]:
import ee

EE_PROJECT = 'ca-morocco'      # must have the Earth Engine API enabled

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print(ee.String('GEE works — project: ' + EE_PROJECT).getInfo())


## 3. Get the 500 sampled fields (data only — the code is in cell 4)


In [ ]:
!mkdir -p prep
!wget -q -O prep/fields_settat_500.geojson \
  https://raw.githubusercontent.com/Floriandbl/notill-validation/main/prep/fields_settat_500.geojson
import json
print(len(json.load(open('prep/fields_settat_500.geojson'))['features']), 'fields loaded')


## 4. The pipeline

Every knob you care about is at the top: `BOX_M`, `SEASON`, `MODE`, the NDTI scale.


In [ ]:
# ============================================================
#  TILLAGE MONTAGES — self-contained pipeline
#  EVERYTHING lives in this cell. The smoke test and the full run call the SAME
#  functions, so any setting you change here actually takes effect. (The old
#  notebook imported a module and then ran a subprocess: the MODE switch silently
#  did nothing and the full run used different settings.)
# ============================================================
import csv, io, json, os, shutil, time
from datetime import date, timedelta
import requests
from PIL import Image, ImageDraw
from pyproj import Transformer

# ---------------- study ----------------
PROVINCE, SEASON   = "Settat", 2025      # season START year: 1 Sep 2025 -> ~22 Dec 2025
SEASON_START_MD    = (9, 1)              # 1 September
N_STEPS, STEP_DAYS = 8, 14               # 8 dates (A..H), every 2 weeks
BOX_M, PANEL_PX    = 650, 320            # 650 m box -> ~0.49 px/m -> median field ~43 px
MAX_CLOUD          = 60
UTM_CRS            = "EPSG:32629"        # UTM 29N — Settat

# ---------------- what to render ----------------
# 'agri'    : B11/B8/B2 false colour        — pretty, but tillage is subtle
# 'residue' : B12/B11/B8                    — residue absorbs B12, so it separates
# 'ndti'    : NDTI on a fixed ramp          — THE COLOUR IS THE RESIDUE AMOUNT
MODE = 'ndti'

AGRI_MIN, AGRI_MAX = [2000, 2900, 250], [5200, 5100, 5000]
RES_MIN,  RES_MAX  = [1000, 2000, 2900], [4500, 5200, 5100]

# MEASURED on your fields (run calibrate_ndti below). Observed: p2 ~0.03-0.13,
# p50 ~0.07-0.16, p98 ~0.31-0.38 — so the earlier 0.00-0.15 guess SATURATED.
# Scale to the SIGNAL (the seasonal median swing ~0.07 -> ~0.16), not the full
# pixel spread, or the interesting range gets squashed into a few shades.
NDTI_MIN, NDTI_MAX = 0.02, 0.25
NDTI_PALETTE = ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']
#   brown = LOW  NDTI = bare soil        = TILLED
#   teal  = HIGH NDTI = residue retained = NO-TILL
# Keep the scale FIXED across all 8 dates, or a colour change stops meaning a
# ground change — and change over time is the entire signal.

# ---------------- overlay ----------------
# magenta: absent from soil (red), vegetation (green) and the brown->teal ramp.
OUTLINE, OUTLINE_WIDTH, DOT_R = (255, 0, 255), 3, 4.0

GEOJSON    = "prep/fields_settat_500.geojson"
IMAGES_DIR = "images"
META_CSV   = "pairs_metadata.csv"
PANEL_LETTERS = "ABCDEFGHIJKL"
TO_UTM = Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True)

# ---------------- Earth Engine ----------------
def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    return img.updateMask(bad.Not())

def season_windows(year):
    start = date(year, *SEASON_START_MD)
    out = []
    for i in range(N_STEPS):
        s = start + timedelta(days=i * STEP_DAYS)
        e = s + timedelta(days=STEP_DAYS)
        out.append((s.isoformat(), e.isoformat(),
                    f"{PANEL_LETTERS[i]} · {s.strftime('%d %b').lstrip('0')}"))
    return out

def field_rect(feature):
    lon, lat = feature["properties"]["lon"], feature["properties"]["lat"]
    cx, cy = TO_UTM.transform(lon, lat)
    h = BOX_M / 2.0
    bounds = (cx - h, cy - h, cx + h, cy + h)
    return ee.Geometry.Rectangle(list(bounds), UTM_CRS, False), bounds, (cx, cy), lon, lat

def composite(rect, start, end):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(rect).filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
            .map(mask_s2_scl).median())

def visualize_comp(comp):
    """Reads the CURRENT global MODE — so changing MODE really changes the output."""
    if MODE == 'agri':
        return comp.visualize(bands=["B11", "B8", "B2"], min=AGRI_MIN, max=AGRI_MAX)
    if MODE == 'residue':
        return comp.visualize(bands=["B12", "B11", "B8"], min=RES_MIN, max=RES_MAX)
    return comp.normalizedDifference(["B11", "B12"]).visualize(
        min=NDTI_MIN, max=NDTI_MAX, palette=NDTI_PALETTE)

def fetch_panel(rect, start, end, tries=5):
    url = visualize_comp(composite(rect, start, end)).getThumbURL({
        "region": rect, "dimensions": f"{PANEL_PX}x{PANEL_PX}",   # exact -> exact m->px
        "crs": UTM_CRS, "format": "jpg"})
    for a in range(tries):
        try:
            r = requests.get(url, timeout=90)
            if r.status_code == 200:
                return Image.open(io.BytesIO(r.content)).convert("RGB")
        except requests.RequestException:
            pass
        time.sleep(2 ** a)
    raise RuntimeError("thumbnail download failed")

# ---------------- calibration ----------------
def calibrate_ndti(n_fields=5, lo=5, hi=95, apply=False):
    """Measure NDTI across fields x all 8 dates and suggest a FIXED scale."""
    feats = json.load(open(GEOJSON))["features"][:n_fields]
    los, his = [], []
    for f in feats:
        rect, _, _, _, _ = field_rect(f)
        for start, end, lab in season_windows(SEASON):
            nd = composite(rect, start, end).normalizedDifference(["B11", "B12"])
            st = nd.reduceRegion(ee.Reducer.percentile([lo, hi]), rect, 10,
                                 maxPixels=1e9).getInfo()
            a, b = st.get(f"nd_p{lo}"), st.get(f"nd_p{hi}")
            if a is not None and b is not None:
                los.append(a); his.append(b)
    if not los:
        print("no NDTI samples (all cloudy?)"); return
    print(f"NDTI over {len(feats)} fields x {N_STEPS} dates")
    print(f"  p{lo}: {min(los):.3f} .. {max(los):.3f}")
    print(f"  p{hi}: {min(his):.3f} .. {max(his):.3f}")
    lo_v, hi_v = round(min(los), 2), round(max(his), 2)
    print(f"\n  suggested:  NDTI_MIN, NDTI_MAX = {lo_v}, {hi_v}")
    if apply:
        globals()["NDTI_MIN"], globals()["NDTI_MAX"] = lo_v, hi_v
        print("  applied.")

# ---------------- local overlay + montage ----------------
def overlay_field(img, ring_utm, cxy, bounds):
    xmin, ymin, xmax, ymax = bounds
    W, H = img.size
    def to_px(X, Y):
        return ((X - xmin) / (xmax - xmin) * W, (ymax - Y) / (ymax - ymin) * H)
    d = ImageDraw.Draw(img)
    pts = [to_px(X, Y) for X, Y in ring_utm]
    if len(pts) > 1:
        # joint="curve" closes the corner gaps a thick polyline leaves
        d.line(pts + [pts[0]], fill=OUTLINE, width=OUTLINE_WIDTH, joint="curve")
    cx, cy = to_px(*cxy)
    d.ellipse([cx - DOT_R, cy - DOT_R, cx + DOT_R, cy + DOT_R],
              fill=OUTLINE, outline=(255, 255, 255))
    return img

def make_montage(panels, labels):
    cols, rows, pad, cap = 4, 2, 5, 20        # 8 facets: 4 across x 2 down
    w, h = panels[0].size
    canvas = Image.new("RGB", (cols*w + (cols+1)*pad, rows*(h+cap) + (rows+1)*pad),
                       (245, 243, 237))
    draw = ImageDraw.Draw(canvas)
    for i, (im, lab) in enumerate(zip(panels, labels)):
        c, r = i % cols, i // cols
        x = pad + c*(w+pad); y = pad + r*(h+cap+pad)
        draw.text((x+2, y+4), lab, fill=(70, 70, 70))
        canvas.paste(im, (x, y+cap))
    return canvas

# ---------------- runner ----------------
def process(feature, windows, out_root=None):
    out_root = out_root or IMAGES_DIR
    rect, bounds, cxy, lon, lat = field_rect(feature)
    ring_utm = [TO_UTM.transform(x, y) for x, y in feature["geometry"]["coordinates"][0]]
    panels, labels = [], []
    for start, end, label in windows:
        panels.append(overlay_field(fetch_panel(rect, start, end), ring_utm, cxy, bounds))
        labels.append(label)
    fname = f"{lat:.5f}_{lon:.5f}_{SEASON}_{PROVINCE}.jpg"
    rel = f"{PROVINCE}/{SEASON}/{fname}"
    out = os.path.join(out_root, PROVINCE, str(SEASON), fname)
    os.makedirs(os.path.dirname(out), exist_ok=True)
    make_montage(panels, labels).save(out, quality=88)
    return rel, lon, lat

def run_all(limit=None, clean=True, tag=None):
    """tag: subfolder for A/B runs, so comparisons don't overwrite each other."""
    img_dir = IMAGES_DIR if tag is None else os.path.join(IMAGES_DIR, "_ab", tag)
    if clean:
        shutil.rmtree(img_dir, ignore_errors=True)
        if tag is None and os.path.exists(META_CSV): os.remove(META_CSV)
    feats = json.load(open(GEOJSON))["features"]
    if limit: feats = feats[:limit]
    windows = season_windows(SEASON)
    scale = (f"NDTI {NDTI_MIN}..{NDTI_MAX}" if MODE == 'ndti' else
             f"{AGRI_MIN}..{AGRI_MAX}" if MODE == 'agri' else f"{RES_MIN}..{RES_MAX}")
    print(f"MODE={MODE} | {scale} | BOX_M={BOX_M} m | SEASON={SEASON}")
    print(f"{len(feats)} fields x {N_STEPS} dates = {len(feats)*N_STEPS} thumbnails\n")
    rows = []
    for i, f in enumerate(feats, 1):
        try:
            rel, lon, lat = process(f, windows, out_root=img_dir)
            rows.append({"pair_id": f["properties"]["field_id"], "province": PROVINCE,
                         "year": SEASON, "image_a": rel, "image_b": "",
                         "lat_a": lat, "lon_a": lon, "lat_b": "", "lon_b": ""})
            print(f"  [{i}/{len(feats)}] {rel}")
        except Exception as e:
            print(f"  [{i}/{len(feats)}] SKIPPED ({e})")
    if tag is None:
        cols = ["pair_id","province","year","image_a","image_b","lat_a","lon_a","lat_b","lon_b"]
        new = not os.path.exists(META_CSV)
        with open(META_CSV, "a", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            if new: w.writerow(cols)
            for r in rows: w.writerow([r[c] for c in cols])
    print(f"\nDone. {len(rows)} fields -> {img_dir}")
    return rows

def show_montages(n=1, tag=None):
    import glob
    from IPython.display import Image as Show, display
    root = IMAGES_DIR if tag is None else os.path.join(IMAGES_DIR, "_ab", tag)
    for p in sorted(glob.glob(f"{root}/**/*.jpg", recursive=True))[:n]:
        print(p); display(Show(p))

print("pipeline loaded.  next:  calibrate_ndti(apply=True)  ->  compare_modes()")


## 5. Calibrate the NDTI scale on YOUR data

Never guess a stretch — that bug has bitten twice now (`max=3000` made everything yellow;
`NDTI_MAX=0.15` would have saturated the teal end). `apply=True` sets the globals for you.


In [ ]:
calibrate_ndti(n_fields=5, apply=True)


## 6. Render every mode for every field, and keep them

`save_all_modes()` renders each field in **all three** modes into its own folder
(`images/_ab/agri|residue|ndti/`), and `stack_modes()` builds one combined image per
field so you can compare the three renderings side by side.

**Pick the mode where tillage is most obvious**, then set `MODE` in cell 4 and re-run it.


In [ ]:
# Render the SAME fields in EVERY mode and KEEP all versions.
# Each mode writes to images/_ab/<mode>/... so nothing overwrites anything.
# Cost: n_fields x 8 dates x 3 modes thumbnails (3 fields -> 72).

def save_all_modes(n_fields=3, modes=('agri', 'residue', 'ndti')):
    """Every field rendered in every mode. Returns {mode: [relative paths]}."""
    global MODE
    keep = MODE
    made = {}
    try:
        for m in modes:
            MODE = m          # fetch_panel reads this global -> the switch really applies
            print(f"\n{'='*22} MODE: {m} {'='*22}")
            made[m] = [r["image_a"] for r in run_all(limit=n_fields, tag=m)]
    finally:
        MODE = keep
    print(f"\nMODE restored to {MODE!r}")
    return made


def stack_modes(modes=('agri', 'residue', 'ndti'), show=True):
    """One combined image PER FIELD: the three renderings stacked and labelled,
    so you can compare them at a glance (and download/share a single file)."""
    import glob
    per_mode = {m: sorted(glob.glob(f"{IMAGES_DIR}/_ab/{m}/**/*.jpg", recursive=True))
                for m in modes}
    n = min((len(v) for v in per_mode.values()), default=0)
    if n == 0:
        print("nothing to stack — run save_all_modes() first"); return []
    outdir = os.path.join(IMAGES_DIR, "_ab", "compare")
    os.makedirs(outdir, exist_ok=True)
    out = []
    for i in range(n):
        ims = [Image.open(per_mode[m][i]).convert("RGB") for m in modes]
        lab_h = 22
        W = max(im.width for im in ims)
        H = sum(im.height + lab_h for im in ims)
        canvas = Image.new("RGB", (W, H), (245, 243, 237))
        d = ImageDraw.Draw(canvas)
        y = 0
        for m, im in zip(modes, ims):
            d.text((4, y + 5), m.upper(), fill=(20, 20, 20))
            canvas.paste(im, (0, y + lab_h))
            y += im.height + lab_h
        name = os.path.basename(per_mode[modes[0]][i]).replace(".jpg", "")
        p = os.path.join(outdir, f"{name}_modes.jpg")
        canvas.save(p, quality=88)
        out.append(p)
        print("wrote", p)
        if show:   # imported here so the function also works outside Colab
            from IPython.display import Image as Show, display
            display(Show(p))
    return out


save_all_modes(n_fields=3)     # 3 fields x 3 modes, all kept
stack_modes()                  # one stacked comparison image per field


### What to look for

- **`ndti`** — a field that stays **teal** all season = residue kept = **no-till**.
  A field that flips **teal → brown** = residue buried = **tilled**, and *that date is the*
  *answer to the follow-up question*.
- Is the magenta delineation on the right field?
- Does `H · 8 Dec` still carry signal, or has it gone flat?

Reality check: at 10 m your median field is ~9 pixels. If none of the three modes makes
tillage legible, the answer is **better imagery** (PlanetScope 3 m), not more colour tuning.


## 7. Full run — 500 fields x 8 dates (~20-40 min)


In [ ]:
run_all()          # same code as the smoke test, so what you saw is what you get


## 8. Download


In [ ]:
!zip -qr montages.zip images pairs_metadata.csv
!du -h montages.zip
from google.colab import files
files.download('montages.zip')


Then locally: unzip into the App folder (replacing `images/` and `pairs_metadata.csv`),
`Rscript r/build_pairs.R`, load `pairs_seed.sql` into Supabase, `git push`.
